# need to make threads to be 8 ..

In [ ]:
using Base
ENV["JULIA_NUM_THREADS"] = "auto"
Threads.nthreads()

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n#Pkg.develop(path="/Users/nobuaki/Documents/Github/sonifSismo.jl")
#Pkg.instantiate()\n

In [ ]:



include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))

using .commonBatchs, .planet1D, .GeoPoints
using Colors



In [ ]:
using Statistics

In [ ]:
using GLMakie
GLMakie.activate!()
Makie.inline!()

In [ ]:
p0 = GeoPoint(35.363602,138.726379) # summit of Mt Fuji


In [ ]:


Δx = 300.0 # in metre
Δy = 300.0
Δz = 10.0

altMax = 4.e3 # in metre
altMin = -10.e0 # in metre

horizontalDepth = 20.e3 

In [ ]:
boxGrids3D=constructLocalBox(p0,Δx,Δy,Δz,-horizontalDepth,horizontalDepth,-horizontalDepth,horizontalDepth,altMin,altMax)

In [ ]:
earthquakePoints = [GeoPoint(35.35,138.71),GeoPoint(35.36,138.72)]
stationPoints    = [GeoPoint(35.351,138.712),GeoPoint(35.361,138.721,alt=-3.e3)]  # array of GeoPoint


In [ ]:

extraPointSets = [
    GeoPointSet("earthquakes", earthquakePoints; color=:red, marker=:star5),
    GeoPointSet("stations", stationPoints; color=:blue, marker=:utriangle),
]

In [ ]:
eqLocal = [
    p_ECEF_to_local(p.ecef, p0.ecef, boxGrids3D.rotationMatrix)
    for p in earthquakePoints
]

staLocal = [
    p_ECEF_to_local(p.ecef, p0.ecef, boxGrids3D.rotationMatrix)
    for p in stationPoints
]

In [ ]:
eqLocal = GeoPoints_to_local(earthquakePoints, boxGrids3D)
staLocal = GeoPoints_to_local(stationPoints, boxGrids3D)

In [ ]:
seismicModel3D=lazyProduceOrLoad("seismicModel3D_Fuji",getParamsAndTopo,boxGrids3D.allGridsInGeoPoints,boxGrids3D.effectiveRadii,0.5)

In [ ]:
Nx3D,Ny3D,Nz3D=boxGrids3D.Nx, boxGrids3D.Ny, boxGrids3D.Nz

In [ ]:
boxGrids3D.allGridsInGeoPoints[1,1,1]

In [ ]:


x = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:,1,1]]*1.e-3
y = [p.xyz[2] for p in boxGrids3D.allGridsInCartesian[1,:,1]]*1.e-3
z = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1,1,:]]*1.e-3


ρ = seismicModel3D.ρ

sx = max(1, ceil(Int, size(ρ, 1) / 512))
sy = max(1, ceil(Int, size(ρ, 2) / 512))
sz = max(1, ceil(Int, size(ρ, 3) / 256))

ρ_plot = ρ[1:sx:end, 1:sy:end, 1:sz:end]
xg = x[1:sx:end]
yg = y[1:sy:end]
zg = z[1:sz:end]

println("original size = ", size(ρ))
println("plot size = ", size(ρ_plot))
println("strides = ", (sx, sy, sz))


In [ ]:




f = Figure()
ax = Axis3(f[1, 1])

volume!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    ρ_plot;
    algorithm = :absorption,
    colormap = :viridis,
)
f

# finding vertical gradient of Vpv

In [ ]:
Vp = seismicModel3D.Vpv

# Vertical Vp gradient, same size as Vp
dVp_dz = similar(Vp)

for k in 2:length(z)-1
    dVp_dz[:, :, k] .= (Vp[:, :, k+1] .- Vp[:, :, k-1]) ./ (z[k+1] - z[k-1])
end

dVp_dz[:, :, 1] .= (Vp[:, :, 2] .- Vp[:, :, 1]) ./ (z[2] - z[1])
dVp_dz[:, :, end] .= (Vp[:, :, end] .- Vp[:, :, end-1]) ./ (z[end] - z[end-1])
Gg = dVp_dz[1:sx:end, 1:sy:end, 1:sz:end];

In [ ]:
valid = filter(isfinite, vec(abs.(Gg)))
thr = quantile(valid, 0.993)   # strongest 2%
println("threshold = ", thr)
thr=8

In [ ]:
Pkg.add("MarchingCubes")
using MarchingCubes

In [ ]:
f = Figure()
ax = Axis3(f[1, 1];
    xlabel = "x [km]",
    ylabel = "y [km]",
    zlabel = "z [km]",
)

contour!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    abs.(Gg);
    levels = [thr],
    colormap = [:orange],
    alpha = 0.6,
    transparency = true,
    linewidth = 0,
)
f

In [ ]:
using CSV
using DataFrames
using GLMakie

In [ ]:
stations_df = CSV.read(
    "/Users/nobuaki/Documents/Github/sonifSismo.jl/exports/stations_available.csv",
    DataFrame,
)

events_df = CSV.read(
    "/Users/nobuaki/Documents/Github/sonifSismo.jl/exports/events_selected.csv",
    DataFrame,
)

println(names(stations_df))
println(names(events_df))

In [ ]:
function findcol(df::DataFrame, candidates)
    name_map = Dict(lowercase(String(n)) => n for n in names(df))

    for c in candidates
        key = lowercase(String(c))
        if haskey(name_map, key)
            return name_map[key]
        end
    end

    error("Could not find $(candidates). Available columns: $(names(df))")
end

function nonmissing_rows(df::DataFrame, cols)
    mask = trues(nrow(df))
    for col in cols
        mask .&= .!ismissing.(df[:, col])
    end
    return df[mask, :]
end

In [ ]:
sta_lat_col = findcol(stations_df, [:lat, :latitude])
sta_lon_col = findcol(stations_df, [:lon, :longitude])

sta_alt_col = try
    findcol(stations_df, [:alt, :altitude, :elevation, :elev, :altitude_m])
catch
    nothing
end

stations_valid = nonmissing_rows(stations_df, [sta_lat_col, sta_lon_col])

stationPoints = [
    GeoPoint(
        Float64(row[sta_lat_col]),
        Float64(row[sta_lon_col]);
        alt=
        sta_alt_col === nothing || ismissing(row[sta_alt_col]) ?
            0.0 :
            Float64(row[sta_alt_col]),
    )
    for row in eachrow(stations_valid)
]

In [ ]:
stationPoints

In [ ]:
ev_lat_col = findcol(events_df, [:lat, :latitude])
ev_lon_col = findcol(events_df, [:lon, :longitude])

ev_depth_col = try
    findcol(events_df, [:depth, :depth_km, :dep])
catch
    nothing
end

ev_type_col = try
    findcol(events_df, [:type, :event_type, :kind, :class, :label])
catch
    nothing
end

events_valid = nonmissing_rows(events_df, [ev_lat_col, ev_lon_col])

eventPoints = [
    GeoPoint(
        Float64(row[ev_lat_col]),
        Float64(row[ev_lon_col]);
        alt=
        ev_depth_col === nothing || ismissing(row[ev_depth_col]) ?
            0.0 :
            -Float64(row[ev_depth_col]) * 1e3,
    )
    for row in eachrow(events_valid)
]

In [ ]:
function GeoPoints_to_local(points::AbstractVector{GeoPoint}, boxGrids3D)
    pOrigin = boxGrids3D.pOriginECEF 
    R = boxGrids3D.rotationMatrix

    return [
        p_ECEF_to_local(p.ecef, pOrigin, R)
        for p in points
    ]
end


In [ ]:
eqLocal = GeoPoints_to_local(eventPoints, boxGrids3D)
staLocal = GeoPoints_to_local(stationPoints, boxGrids3D)

In [ ]:
eventTypes = ev_type_col === nothing ?
    fill("event", nrow(events_valid)) :
    [
        ismissing(row[ev_type_col]) ? "unknown" : lowercase(String(row[ev_type_col]))
        for row in eachrow(events_valid)
    ]

In [ ]:
function local_xyz_km(localPoints)
    xs = getindex.(localPoints, 1) .* 1e-3
    ys = getindex.(localPoints, 2) .* 1e-3
    zs = getindex.(localPoints, 3) .* 1e-3
    return xs, ys, zs
end

eqX, eqY, eqZ = local_xyz_km(eqLocal)
staX, staY, staZ = local_xyz_km(staLocal)

In [ ]:
staX, staY, staZ

In [ ]:
f = Figure()
ax = Axis3(f[1, 1];
    xlabel = "eastward [km]",
    ylabel = "northward [km]",
    zlabel = "altitude [km]",
)

contour!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    abs.(Gg);
    levels = [thr],
    color = (:orange, 0.45),
    transparency = true,
    linewidth = 0,
)


scatter!(
    ax,
    staX, staY, staZ;
    color = :dodgerblue,
    marker = :utriangle,
    markersize = 14,
    label = "stations",
)



axislegend(ax)
#limits!(ax, -200, 200, -200, 200, -400, 5)
f

In [ ]:
f = Figure()
ax = Axis3(f[1, 1];
    xlabel = "eastward [km]",
    ylabel = "northward [km]",
    zlabel = "altitude [km]",
)

contour!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    abs.(Gg);
    levels = [thr],
    color = (:orange, 0.45),
    transparency = true,
    linewidth = 0,
)


scatter!(
    ax,
    staX, staY, staZ;
    color = :dodgerblue,
    marker = :utriangle,
    markersize = 14,
    label = "stations",
)
scatter!(
    ax,
    eqX, eqY, eqZ;
    color = :red,
    marker = :circle,
    markersize = 7,
    label = "events",
)




axislegend(ax)
#limits!(ax, -200, 200, -200, 200, -400, 5)
f

In [ ]:
maximum(eqZ)